# TP — Grover's Search Algorithm

Ce notebook suit les deux parties du sujet :

1. **Grover sur 2 qubits** avec une seule itération et une cible dynamique.
2. **Grover sur 3 qubits** avec un nombre d'itérations \(r\) variable de 1 à 20.

Le code est volontairement séparé en deux parties pour être plus simple à lire et à expliquer.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram


# Partie 1 — Grover sur 2 qubits

Pour 2 qubits :

\[
N = 2^2 = 4
\]

Les quatre états possibles sont :

\[
|00\rangle,\quad |01\rangle,\quad |10\rangle,\quad |11\rangle
\]

On réalise exactement **une itération de Grover**.


## 1. Oracle à 2 qubits


In [ ]:
def oracle_2q(qc, target):

    for q, bit in enumerate(reversed(target)):
        if bit == "0":
            qc.x(q)

    # CZ change la phase de |11>
    qc.cz(0, 1)

    # On annule les portes X précédentes
    for q, bit in enumerate(reversed(target)):
        if bit == "0":
            qc.x(q)


## 2. Opérateur de diffusion à 2 qubits


In [ ]:
def diffusion_2q(qc):
    """
    Diffusion de Grover pour 2 qubits.
    """

    qc.h([0, 1])
    qc.x([0, 1])

    qc.cz(0, 1)

    qc.x([0, 1])
    qc.h([0, 1])


## 3. Circuit complet de Grover à 2 qubits


In [ ]:
def grover_2q(target):
    """
    Circuit complet de Grover sur 2 qubits
    avec exactement une itération.
    """

    qc = QuantumCircuit(2)

    # Étape 1 : superposition uniforme
    qc.h([0, 1])

    qc.barrier()

    # Étape 2 : oracle
    oracle_2q(qc, target)

    qc.barrier()

    # Étape 3 : diffusion
    diffusion_2q(qc)

    qc.barrier()

    # Étape 4 : mesure
    qc.measure_all()

    return qc


## 4. Choix de la cible et affichage du circuit


In [ ]:
# Changer uniquement cette valeur pour tester une autre cible :
# "00", "01", "10" ou "11"
target_2q = "10"

qc_2q = grover_2q(target_2q)

qc_2q.draw()


## 5. Simulation avec 1024 shots


In [ ]:
simulator = AerSimulator(seed_simulator=42)

compiled_2q = transpile(qc_2q, simulator)

result_2q = simulator.run(
    compiled_2q,
    shots=1024
).result()

counts_2q = result_2q.get_counts()

print("Cible :", target_2q)
print("Résultats :", counts_2q)

plot_histogram(
    counts_2q,
    title=f"Grover — 2 qubits — cible = {target_2q}"
)


### Commentaire — Partie 1

Pour \(N=4\), une seule itération de Grover est optimale.

Dans un simulateur idéal, l'état cible doit donc apparaître avec une probabilité très proche de 1.

Par exemple, pour la cible `10`, on s'attend idéalement à obtenir :

```text
{'10': 1024}
```

L'oracle change la phase de l'état recherché, puis l'opérateur de diffusion amplifie son amplitude.


# Partie 2 — Grover sur 3 qubits

Pour 3 qubits :

\[
N = 2^3 = 8
\]

Le sujet propose par exemple la cible :

\[
x^* = 101
\]

Cette fois, le nombre d'itérations \(r\) est variable.


## 6. Oracle à 3 qubits


In [ ]:
def oracle_3q(qc, target):
    """
    Oracle de Grover pour 3 qubits.
    target doit être une chaîne binaire de longueur 3.
    """

    if len(target) != 3 or any(bit not in "01" for bit in target):
        raise ValueError("La cible doit contenir exactement 3 bits, par exemple 101.")

    # Transformer temporairement la cible en |111>
    for q, bit in enumerate(reversed(target)):
        if bit == "0":
            qc.x(q)

    # CCZ = H sur le troisième qubit + CCX + H
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

    # Annuler les portes X précédentes
    for q, bit in enumerate(reversed(target)):
        if bit == "0":
            qc.x(q)


## 7. Opérateur de diffusion à 3 qubits


In [ ]:
def diffusion_3q(qc):
    """
    Diffusion de Grover pour 3 qubits.
    """

    qc.h([0, 1, 2])
    qc.x([0, 1, 2])

    # CCZ
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

    qc.x([0, 1, 2])
    qc.h([0, 1, 2])


## 8. Circuit complet de Grover à 3 qubits


In [ ]:
def grover_3q(target, r):
    """
    Circuit de Grover sur 3 qubits.
    r = nombre d'itérations de Grover.
    """

    if r < 0:
        raise ValueError("r doit être positif ou nul.")

    qc = QuantumCircuit(3)

    # Étape 1 : superposition uniforme
    qc.h([0, 1, 2])

    # Étape 2 : r itérations de Grover
    for _ in range(r):

        qc.barrier()

        oracle_3q(qc, target)

        qc.barrier()

        diffusion_3q(qc)

    qc.barrier()

    # Étape 3 : mesure
    qc.measure_all()

    return qc


## 9. Exemple avec une valeur de r


In [ ]:
target_3q = "101"
r = 2

qc_3q = grover_3q(target_3q, r)

qc_3q.draw()


## 10. Simulation de cet exemple


In [ ]:
compiled_3q = transpile(qc_3q, simulator)

result_3q = simulator.run(
    compiled_3q,
    shots=1024
).result()

counts_3q = result_3q.get_counts()

print("Cible :", target_3q)
print("Nombre d'itérations :", r)
print("Résultats :", counts_3q)

plot_histogram(
    counts_3q,
    title=f"Grover — 3 qubits — cible = {target_3q}, r = {r}"
)


# 11. Probabilité théorique

Le sujet donne :

\[
P(r)
=
\sin^2\left[
\left(r+\frac{1}{2}\right)\theta
\right]
\]

avec

\[
\theta
=
2\arcsin\left(\frac{1}{\sqrt{N}}\right)
\]

Ici :

\[
N=8
\]


In [ ]:
N = 8

theta = 2 * np.arcsin(1 / np.sqrt(N))

print("theta =", theta)


## 12. Simulation pour r = 1 à 20


In [ ]:
target_3q = "101"
shots = 1024

r_values = range(1, 21)

p_empirical = []
p_theory = []

for r in r_values:

    # Circuit pour la valeur de r
    qc = grover_3q(target_3q, r)

    # Simulation
    compiled = transpile(qc, simulator)

    result = simulator.run(
        compiled,
        shots=shots
    ).result()

    counts = result.get_counts()

    # Probabilité expérimentale
    p_exp = counts.get(target_3q, 0) / shots
    p_empirical.append(p_exp)

    # Probabilité théorique
    p_th = np.sin((r + 0.5) * theta) ** 2
    p_theory.append(p_th)

    print(
        f"r = {r:2d}   "
        f"P_exp = {p_exp:.4f}   "
        f"P_th = {p_th:.4f}"
    )


## 13. Graphique : théorie et simulation


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    list(r_values),
    p_theory,
    marker="o",
    label="Probabilité théorique"
)

plt.scatter(
    list(r_values),
    p_empirical,
    label="Probabilité expérimentale"
)

plt.xlabel("Nombre d'itérations r")
plt.ylabel("Probabilité de succès")
plt.title("Algorithme de Grover — 3 qubits — cible 101")

plt.xticks(range(1, 21))
plt.ylim(0, 1.05)

plt.grid()
plt.legend()

plt.show()


## 14. Calcul du nombre optimal d'itérations


In [ ]:
r_opt = int(np.floor((np.pi / 4) * np.sqrt(N)))

print("Nombre optimal d'itérations selon la formule du sujet :", r_opt)

for r_test in [1, 2, 3]:
    p = np.sin((r_test + 0.5) * theta) ** 2
    print(f"r = {r_test}  ->  P_th = {p:.6f}")


### Commentaire — Partie 2

La probabilité de mesurer l'état cible n'augmente pas continuellement avec le nombre d'itérations.

Elle **oscille** avec \(r\).

Pour \(N=8\) :

\[
r_{\mathrm{opt}}
=
\left\lfloor
\frac{\pi}{4}\sqrt{8}
\right\rfloor
=
2
\]

Les premières probabilités théoriques sont approximativement :

\[
P(1)=0.78125
\]

\[
P(2)=0.9453125
\]

\[
P(3)=0.3300781
\]

Le premier maximum est donc obtenu pour :

\[
\boxed{r=2}
\]

avec une probabilité de succès d'environ \(94.5\%\).

Si on continue à appliquer l'opérateur de Grover après ce maximum, la probabilité diminue puis augmente à nouveau. Cela explique l'allure oscillante de la courbe.


# Conclusion

L'algorithme de Grover permet de rechercher un élément dans une base non triée en environ :

\[
O(\sqrt{N})
\]

appels à l'oracle, contre \(O(N)\) pour une recherche classique.

- Avec **2 qubits**, une seule itération permet idéalement de retrouver la cible avec une probabilité égale à 1.
- Avec **3 qubits**, la probabilité de succès dépend du nombre d'itérations.
- Pour \(N=8\), le premier maximum est obtenu pour \(r=2\).
